# Mistral-7B Text Generation — JAX Optimized

This notebook is **Phase 5** of the LLM Response Time Optimizer project.
It validates end-to-end text generation with Mistral-7B using our JAX pipeline
and benchmarks it against the PyTorch baseline.

**What this notebook does:**
1. Loads and converts Mistral-7B (PyTorch → JAX)
2. Runs a single generation test to confirm the pipeline works
3. Benchmarks the same 3 prompts used in the PyTorch baseline
4. Computes tokens/sec, latency, and speedup vs baseline

**PyTorch Baseline Results (from `01_baseline_pytorch.ipynb`):**
| Prompt | PyTorch Latency |
|--------|-----------------|
| Explain quantum computing | 13.69s |
| Write a fibonacci function | 18.08s |
| What is machine learning? | 42.28s |
| **Average** | **24.68s** |

**Requirements:**
- Google Colab with GPU (T4 or better recommended)
- ~14GB RAM for model weights
- Project repo cloned (or uploaded manually)

## 1. Setup Environment

In [1]:
# Check environment
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Check for GPU
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL).decode()
    print("GPU detected:")
    # Print just the GPU name line
    for line in gpu_info.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'V100' in line or 'A100' in line or 'RTX' in line:
            print(' ', line.strip())
except Exception:
    print("No GPU detected — generation will be slow on CPU")

Running in Google Colab
GPU detected:
  |   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |


In [2]:
# Install dependencies if needed
# Uncomment whichever packages are missing in your environment

!pip install -q torch transformers
!pip install -q jax[cuda12] jaxlib
!pip install -q flax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 56.2 MB/s eta 0:00:00


In [3]:
# Clone the project repo from GitHub
# This gives us the latest src/ code (with all Mistral fixes applied)
!git clone https://github.com/YashM246/LLM_Response_Time_Optimizer.git

import sys
sys.path.append('./LLM_Response_Time_Optimizer/')

Cloning into 'LLM_Response_Time_Optimizer'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 350 (delta 55), reused 56 (delta 27), pack-reused 260 (from 1)
Receiving objects: 100% (350/350), 208.20 KiB | 17.35 MiB/s, done.
Resolving deltas: 100% (208/208), done.


In [4]:
import jax
import jax.numpy as jnp
import time
import json

from src.model_conversion import convert_model
from src.cached_generation import generate_text_with_cache, MISTRAL_CONFIG

print(f"JAX version  : {jax.__version__}")
print(f"JAX backend  : {jax.default_backend()}")
print(f"Devices      : {jax.devices()}")
print("Imports OK")

JAX version  : 0.7.2
JAX backend  : gpu
Devices      : [CudaDevice(id=0)]
Imports OK


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load and Convert Mistral-7B

This cell downloads Mistral-7B from HuggingFace (~14GB) and converts
the PyTorch weights to JAX arrays with the correct PyTree structure.

Expected time: **2–5 minutes** on Colab.

In [6]:
print("Loading and converting Mistral-7B...")
print("(This downloads ~14GB — grab a coffee)\n")

load_start = time.time()

params, tokenizer, model_type = convert_model(model_type="mistral")

load_elapsed = time.time() - load_start
print(f"\nModel loaded and converted in {load_elapsed:.1f}s ({load_elapsed/60:.1f} min)")

Loading and converting Mistral-7B...
(This downloads ~14GB — grab a coffee)


PyTorch -> JAX Conversion Pipeline (MISTRAL)

[1/4] Loading PyTorch model...
Loading Mistral-7B model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[OK] Loaded 291 parameters from mistralai/Mistral-7B-v0.1
Example weight keys:
    model.embed_tokens.weight: torch.Size([32000, 4096])
    model.layers.0.self_attn.q_proj.weight: torch.Size([4096, 4096])
    model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 4096])
    model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 4096])
    model.layers.0.self_attn.o_proj.weight: torch.Size([4096, 4096])

[2/4] Converting to JAX arrays...
  [OK] Transposed model.layers.0.self_attn.q_proj.weight: torch.Size([4096, 4096]) -> (4096, 4096)
  [OK] Transposed model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 4096]) -> (4096, 1024)
  [OK] Transposed model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 4096]) -> (4096, 1024)
  [OK] Transposed model.layers.0.self_attn.o_proj.weight: torch.Size([4096, 4096]) -> (4096, 4096)
  [OK] Transposed model.layers.0.mlp.gate_proj.weight: torch.Size([14336, 4096]) -> (4096, 14336)
  [OK] Transposed model.layers.0.mlp.up_proj.weight: torch.

## 3. Quick Sanity Check

Before running the full benchmark, confirm the converted weights
match the expected Mistral-7B architecture.
This is a fast check — no generation yet.

In [7]:
print("=" * 60)
print("Sanity Checks")
print("=" * 60)

checks_passed = 0
checks_total  = 0

def check(label, condition, detail=""):
    global checks_passed, checks_total
    checks_total += 1
    status = "OK" if condition else "FAIL"
    if condition:
        checks_passed += 1
    suffix = f"  ({detail})" if detail else ""
    print(f"  [{status}] {label}{suffix}")

model_p = params['params']['model']
layer_0  = model_p['layers']['0']

# Structure
check("Top-level 'model' key present",    'model'     in params['params'])
check("Top-level 'lm_head' key present",  'lm_head'   in params['params'])
check("embed_tokens present",             'embed_tokens' in model_p)
check("32 transformer layers",            len(model_p['layers']) == 32,
      f"found {len(model_p['layers'])}")
check("Final norm present",               'norm' in model_p)

# Shapes
embed_shape  = model_p['embed_tokens']['embedding'].shape
lm_shape     = params['params']['lm_head']['kernel'].shape
norm_shape   = model_p['norm']['kernel'].shape
q_shape      = layer_0['self_attn']['q_proj']['kernel'].shape
k_shape      = layer_0['self_attn']['k_proj']['kernel'].shape

check("Embedding shape (32000, 4096)",    embed_shape == (32000, 4096), str(embed_shape))
check("LM head shape  (4096, 32000)",     lm_shape    == (4096, 32000), str(lm_shape))
check("Final norm shape (4096,)",         norm_shape  == (4096,),       str(norm_shape))
check("Q-proj shape (4096, 4096) full",   q_shape     == (4096, 4096),  str(q_shape))
check("K-proj shape (4096, 1024) GQA",   k_shape     == (4096, 1024),  str(k_shape))

# Tokenizer
check("Tokenizer vocab size 32000",       len(tokenizer) == 32000,
      f"found {len(tokenizer)}")

print()
print("=" * 60)
if checks_passed == checks_total:
    print(f"ALL CHECKS PASSED ({checks_passed}/{checks_total})")
else:
    print(f"SOME CHECKS FAILED — {checks_passed}/{checks_total} passed")
    print("Review failures before continuing.")
print("=" * 60)

Sanity Checks
  [OK] Top-level 'model' key present
  [OK] Top-level 'lm_head' key present
  [OK] embed_tokens present
  [OK] 32 transformer layers  (found 32)
  [OK] Final norm present
  [OK] Embedding shape (32000, 4096)  ((32000, 4096))
  [OK] LM head shape  (4096, 32000)  ((4096, 32000))
  [OK] Final norm shape (4096,)  ((4096,))
  [OK] Q-proj shape (4096, 4096) full  ((4096, 4096))
  [OK] K-proj shape (4096, 1024) GQA  ((4096, 1024))
  [OK] Tokenizer vocab size 32000  (found 32000)

ALL CHECKS PASSED (11/11)


## 4. First Generation Test

Run a short generation on a simple prompt to confirm the full
forward pass works end-to-end before the benchmark.

**What to look for:**
- No exceptions or shape errors
- Output text that is coherent (not gibberish / repeated tokens)
- A reasonable tokens/sec figure

The first run will be **slow** — JAX JIT compiles each unique
sequence length on first use. Subsequent runs at the same token
count will be fast.

In [8]:
TEST_PROMPT   = "The capital of France is"
MAX_NEW_TOKENS = 20   # Short — just to validate the pipeline

print("Running first generation (JIT compilation happens here — will be slow)...")
print(f"Prompt        : '{TEST_PROMPT}'")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print("-" * 60)

generated_text, stats = generate_text_with_cache(
    params=params,
    tokenizer=tokenizer,
    prompt=TEST_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.0,   # Greedy — deterministic, easiest to verify
    top_k=0,
    use_cache=True,
    model_type="mistral"
)

print("-" * 60)
print(f"\nGenerated text:")
print(f"  {generated_text}")
print(f"\nStats:")
print(f"  Prompt tokens    : {stats['prompt_length']}")
print(f"  Generated tokens : {stats['generated_tokens']}")
print(f"  Time elapsed     : {stats['time_elapsed']:.2f}s")
print(f"  Tokens/sec       : {stats['tokens_per_sec']:.2f}")

Running first generation (JIT compilation happens here — will be slow)...
Prompt        : 'The capital of France is'
Max new tokens: 20
------------------------------------------------------------
Prompt: 'The capital of France is'
Prompt length: 6 tokens

Prefill phase (processing prompt)...
Mode: Token-by-token with cache
[OK] Prefill complete (6 tokens)

Generating 20 new tokens...
  Generated 10/20 tokens...
  Generated 20/20 tokens...
------------------------------------------------------------

Generated text:
  <s> The capital of France is Paris, and it is one of the most visited cities in the world. It is a city that

Stats:
  Prompt tokens    : 6
  Generated tokens : 20
  Time elapsed     : 31.36s
  Tokens/sec       : 0.64


## 5. Warmup Run

JAX JIT compiles a separate function for each unique sequence length.
During autoregressive generation of N tokens, it compiles N different
functions — one per sequence length.

**Critical rule:** The warmup must generate at least as many tokens as
the benchmark run, otherwise the benchmark will include compilation
time and report an artificially slow result.

This warmup pre-compiles all sequence lengths we will use in the benchmark.

In [9]:
# How many tokens to generate in the benchmark
BENCHMARK_TOKENS = 50

print(f"Warming up — generating {BENCHMARK_TOKENS} tokens to pre-compile all shapes...")
print("(This will take a while on first run — subsequent runs will be fast)\n")

warmup_start = time.time()

_, warmup_stats = generate_text_with_cache(
    params=params,
    tokenizer=tokenizer,
    prompt="Warmup prompt for JAX JIT compilation",
    max_new_tokens=BENCHMARK_TOKENS,
    temperature=0.0,
    top_k=0,
    use_cache=True,
    model_type="mistral"
)

warmup_elapsed = time.time() - warmup_start
print(f"Warmup complete in {warmup_elapsed:.1f}s")
print(f"All {BENCHMARK_TOKENS} sequence-length shapes are now compiled.")
print("Benchmark runs below will reflect true runtime (no compile overhead).")

Warming up — generating 50 tokens to pre-compile all shapes...
(This will take a while on first run — subsequent runs will be fast)

Prompt: 'Warmup prompt for JAX JIT compilation'
Prompt length: 11 tokens

Prefill phase (processing prompt)...
Mode: Token-by-token with cache
[OK] Prefill complete (11 tokens)

Generating 50 new tokens...
  Generated 10/50 tokens...
  Generated 20/50 tokens...
  Generated 30/50 tokens...
  Generated 40/50 tokens...
  Generated 50/50 tokens...
Warmup complete in 42.5s
All 50 sequence-length shapes are now compiled.
Benchmark runs below will reflect true runtime (no compile overhead).


## 6. Benchmark — 3 Prompts vs PyTorch Baseline

We run the exact same 3 prompts used in `01_baseline_pytorch.ipynb`
so the results are directly comparable.

Each prompt is run once. For more statistically robust results,
you could increase `NUM_RUNS` and average.

In [13]:
# The same 3 prompts from the PyTorch baseline notebook
BENCHMARK_PROMPTS = [
    "Explain quantum computing in simple terms",
    "Write a python function to calculate fibonacci",
    "What is machine learning?"
]

# PyTorch baseline latencies (from 01_baseline_pytorch.ipynb)
PYTORCH_LATENCIES = [13.69, 18.08, 42.28]

jax_results = []

print("=" * 70)
print(f"Benchmark: {len(BENCHMARK_PROMPTS)} prompts, {BENCHMARK_TOKENS} tokens each")
print("=" * 70)

for i, prompt in enumerate(BENCHMARK_PROMPTS):
    print(f"\nPrompt {i+1}/{len(BENCHMARK_PROMPTS)}: '{prompt}'")
    print("-" * 70)

    generated_text, stats = generate_text_with_cache(
        params=params,
        tokenizer=tokenizer,
        prompt=prompt,
        max_new_tokens=BENCHMARK_TOKENS,
        temperature=0.7,   # Same temperature as baseline
        top_k=50,
        use_cache=True,
        model_type="mistral"
    )

    jax_results.append({
        'prompt': prompt,
        'generated_text': generated_text,
        'latency': stats['time_elapsed'],
        'tokens_per_sec': stats['tokens_per_sec'],
        'generated_tokens': stats['generated_tokens']
    })

    print(f"\nOutput: {generated_text[:200]}{'...' if len(generated_text) > 200 else ''}")
    print(f"Latency      : {stats['time_elapsed']:.2f}s")
    print(f"Tokens/sec   : {stats['tokens_per_sec']:.2f}")
    print(f"vs PyTorch   : {PYTORCH_LATENCIES[i]:.2f}s")

print("\n" + "=" * 70)
print("Benchmark complete.")
print("=" * 70)

Benchmark: 3 prompts, 50 tokens each

Prompt 1/3: 'Explain quantum computing in simple terms'
----------------------------------------------------------------------
Prompt: 'Explain quantum computing in simple terms'
Prompt length: 8 tokens

Prefill phase (processing prompt)...
Mode: Token-by-token with cache
[OK] Prefill complete (8 tokens)

Generating 50 new tokens...
  Generated 10/50 tokens...
  Generated 20/50 tokens...
  Generated 30/50 tokens...
  Generated 40/50 tokens...
  Generated 50/50 tokens...

Output: <s> Explain quantum computing in simple terms. Explain quantum computing in simple terms.

Quantum computing is a type of computing that works on the fundamental building blocks of the universe, known...
Latency      : 12.70s
Tokens/sec   : 3.94
vs PyTorch   : 13.69s

Prompt 2/3: 'Write a python function to calculate fibonacci'
----------------------------------------------------------------------
Prompt: 'Write a python function to calculate fibonacci'
Prompt length: 10 to

## 7. Results vs PyTorch Baseline

Side-by-side comparison of JAX optimized vs PyTorch baseline.

In [14]:
print("=" * 70)
print("Results: JAX Optimized vs PyTorch Baseline")
print("=" * 70)
print(f"{'Prompt':<42} {'PyTorch':>8} {'JAX':>8} {'Speedup':>8}")
print("-" * 70)

speedups = []

for i, (result, pt_latency) in enumerate(zip(jax_results, PYTORCH_LATENCIES)):
    jax_latency = result['latency']
    speedup     = pt_latency / jax_latency
    speedups.append(speedup)

    # Truncate prompt for display
    prompt_short = result['prompt'][:40] + ('...' if len(result['prompt']) > 40 else '')
    print(f"{prompt_short:<42} {pt_latency:>7.2f}s {jax_latency:>7.2f}s {speedup:>7.2f}x")

print("-" * 70)

avg_pt  = sum(PYTORCH_LATENCIES) / len(PYTORCH_LATENCIES)
avg_jax = sum(r['latency'] for r in jax_results) / len(jax_results)
avg_speedup = avg_pt / avg_jax
avg_toks    = sum(r['tokens_per_sec'] for r in jax_results) / len(jax_results)

print(f"{'AVERAGE':<42} {avg_pt:>7.2f}s {avg_jax:>7.2f}s {avg_speedup:>7.2f}x")
print("=" * 70)

print(f"\nSummary:")
print(f"  Average JAX latency    : {avg_jax:.2f}s")
print(f"  Average PyTorch latency: {avg_pt:.2f}s")
print(f"  Average speedup        : {avg_speedup:.2f}x")
print(f"  Average tokens/sec     : {avg_toks:.2f}")

# Check against project target
TARGET_SPEEDUP = 2.5
TARGET_TOKS    = 20.0
print()
print(f"  Target speedup ({TARGET_SPEEDUP}x): {'ACHIEVED' if avg_speedup >= TARGET_SPEEDUP else 'NOT YET'}")
print(f"  Target tok/sec ({TARGET_TOKS}):  {'ACHIEVED' if avg_toks >= TARGET_TOKS else 'NOT YET'}")

Results: JAX Optimized vs PyTorch Baseline
Prompt                                      PyTorch      JAX  Speedup
----------------------------------------------------------------------
Explain quantum computing in simple term...   13.69s   12.70s    1.08x
Write a python function to calculate fib...   18.08s   12.87s    1.41x
What is machine learning?                    42.28s   12.30s    3.44x
----------------------------------------------------------------------
AVERAGE                                      24.68s   12.62s    1.96x

Summary:
  Average JAX latency    : 12.62s
  Average PyTorch latency: 24.68s
  Average speedup        : 1.96x
  Average tokens/sec     : 3.96

  Target speedup (2.5x): NOT YET
  Target tok/sec (20.0):  NOT YET


## 8. Summary

**What we verified in this notebook:**
- Mistral-7B loads and converts correctly from PyTorch → JAX
- The full generation pipeline runs end-to-end without errors
- KV-Cache + JIT delivers measurable speedup over PyTorch

**Architecture used:**
- RMSNorm (instead of LayerNorm)
- Grouped-Query Attention — 32 Q heads, 8 KV heads
- RoPE positional embeddings
- SwiGLU activation in MLP
- INT8 weight quantization (~2x memory reduction)

**Next steps:**
1. Run `04_quality_evaluation.ipynb` — ROUGE-L scores vs PyTorch baseline
2. Run `05_results_visualization.ipynb` — comparison plots
3. Update `README.md` with the actual Mistral results